In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
import joblib
import gradio as gr

MODEL_FILE = 'usa_housing_model.joblib'
DATA_FILE = 'USA_Housing.csv'

# =====================================================================
# 1. MODEL TRAINING & PREPARATION
# =====================================================================
def train_and_save_model():
    print("--- 1. LOADING & PREPROCESSING DATA ---")
    if not os.path.exists(DATA_FILE):
        raise FileNotFoundError(f"'{DATA_FILE}' not found! Please make sure it is in the same folder.")
    
    df = pd.read_csv(DATA_FILE)
    
    # Drop address column if present
    if 'Address' in df.columns:
        df = df.drop(columns=['Address'])
        
    X = df[['Avg. Area Income', 'Avg. Area House Age', 
            'Avg. Area Number of Rooms', 'Avg. Area Number of Bedrooms', 
            'Area Population']]
    y = df['Price']
    
    # Train / Test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    print("--- 2. TRAINING LINEAR REGRESSION MODEL ---")
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # Evaluate model
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    print(f"Model Trained Successfully! R² Accuracy: {r2 * 100:.2f}%, MAE: ${mae:,.2f}")
    
    # Save Model
    joblib.dump(model, MODEL_FILE)
    print(f"Saved model file to '{MODEL_FILE}'\n")
    return model

# Load existing model or train a new one
if os.path.exists(MODEL_FILE):
    print(f"Loading model from '{MODEL_FILE}'...")
    try:
        model = joblib.load(MODEL_FILE)
    except Exception:
        print("Error loading existing model file. Re-training model...")
        model = train_and_save_model()
else:
    model = train_and_save_model()

# =====================================================================
# 2. PREDICTION FUNCTION FOR GRADIO
# =====================================================================
def predict_house_price(income, age, rooms, bedrooms, population):
    input_df = pd.DataFrame([{
        'Avg. Area Income': income,
        'Avg. Area House Age': age,
        'Avg. Area Number of Rooms': rooms,
        'Avg. Area Number of Bedrooms': bedrooms,
        'Area Population': population
    }])
    
    predicted_price = model.predict(input_df)[0]
    return f"${predicted_price:,.2f}"

# =====================================================================
# 3. GRADIO USER INTERFACE (UI)
# =====================================================================
interface = gr.Interface(
    fn=predict_house_price,
    inputs=[
        gr.Number(label="Avg. Area Income ($)", value=65000),
        gr.Slider(minimum=1.0, maximum=15.0, value=6.0, step=0.1, label="Avg. Area House Age (Years)"),
        gr.Slider(minimum=1.0, maximum=12.0, value=6.0, step=0.1, label="Avg. Number of Rooms"),
        gr.Slider(minimum=1.0, maximum=10.0, value=4.0, step=0.1, label="Avg. Number of Bedrooms"),
        gr.Number(label="Area Population", value=32000)
    ],
    outputs=gr.Textbox(label="Estimated House Price"),
    title="🏠 USA House Price Predictor",
    description="Enter neighborhood details to predict estimated house prices using Linear Regression."
)

if __name__ == "__main__":
    interface.launch(share=True)

ModuleNotFoundError: No module named 'gradio'